# 20 · Source, wording, corruption and food–method binding tests

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run controlled development diagnostics before test lock; later stress tests must be prespecified and reported separately.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Execute source/binding security tests

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_claims_and_gates.py')],cwd=REPO,check=True)

## 2. Construct record-level food–method swap diagnostics
These are report perturbations, not real differently prepared meals or causal chemistry experiments.

In [ ]:
from oncoplate.robustness import tuple_swaps
ann=read_table(p['prepared']/'annotations.csv')
examples=[]
for (rid,reviewer),part in ann.groupby(['record_id','reviewer_id']):
    rows=part[['category','subcategory','cooking_style']].to_dict('records')
    swaps=tuple_swaps(rows)
    if swaps:examples.append({'record_id':rid,'reviewer_id':reviewer,'reference_records':rows,'constructed_swaps':swaps})
write_jsonl(p['private']/'binding_swap_diagnostics.jsonl',examples)
print('Diagnostic cases:',len(examples))

## 3. Save label-preserving image-quality perturbations for validation only

In [ ]:
from oncoplate.pipeline import load_study
from oncoplate.robustness import image_perturbations
records,_=load_study(cfg,'joint',stage_images=True)
subset=records[records.split.eq('validation')].head(16)
for row in subset.itertuples():
    image_perturbations(row.image_path,p['local']/'robustness'/row.record_id)
print('Development perturbations written locally. Compare all methods on the same perturbed cases.')

## 4. Preserve strata and reference limits

In [ ]:
print('Do not remove an item from the image and retain an unchanged presence target.')
print('Evaluate stale, incorrect and conflicting metadata separately; do not pool their artificial frequencies into population risk.')
print('Source rows marked reference_only must never reach the inference policy.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
